# 🎵 DC v2.0 — Build Android APK

This notebook builds the DC YouTube MP3 Downloader APK.

**Run each cell in order.** Total build time: ~15 minutes.

## Step 1: Install System Dependencies

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq \
    build-essential git ffmpeg libffi-dev libssl-dev \
    python3-venv zip unzip autoconf automake libtool \
    pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev \
    cmake openjdk-17-jdk lld
print('\n✅ System dependencies installed')

## Step 2: Install Buildozer & Cython

In [ ]:
!pip install -q buildozer==1.5.0 cython==0.29.33
!buildozer --version
!cython --version
print('\n✅ Buildozer & Cython installed')

## Step 3: Clone Repository

In [ ]:
%cd /content
!rm -rf DC
!git clone https://github.com/SONUVERMA11/DC.git
%cd /content/DC/android_app
!ls -la
print('\n✅ Repository cloned')

## Step 4: Build APK ⏳

**This takes 10-20 minutes on first run.** The cell will show live build output.

In [ ]:
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

!java -version
print('\n🔨 Starting build...\n')
!buildozer -v android debug 2>&1 | tail -100
print('\n🏁 Build process completed')

## Step 5: Find & Download APK

In [ ]:
import glob
from google.colab import files

# Search for APK
apk_files = glob.glob('/content/DC/android_app/bin/*.apk')
if not apk_files:
    apk_files = glob.glob('/content/DC/android_app/.buildozer/**/outputs/apk/**/*.apk', recursive=True)

if apk_files:
    for apk in apk_files:
        size_mb = os.path.getsize(apk) / (1024 * 1024)
        print(f'✅ Found APK: {apk} ({size_mb:.1f} MB)')
        files.download(apk)
else:
    print('❌ No APK found. Checking build log...')
    !tail -50 /content/DC/android_app/build.log 2>/dev/null || echo 'No build log found'
    print('\nSearching all APK files...')
    !find /content/DC -name '*.apk' -type f 2>/dev/null

## 🔍 Debug (optional — run only if build fails)

In [ ]:
# Show last 200 lines of build log
!tail -200 /content/DC/android_app/.buildozer/logs/buildozer.log 2>/dev/null || echo 'Checking build.log...'
!tail -200 /content/DC/android_app/build.log 2>/dev/null || echo 'No logs found'
print('\n--- ERRORS ---')
!grep -i -E '(error|exception|failed|traceback)' /content/DC/android_app/.buildozer/logs/buildozer.log 2>/dev/null | tail -30